# Datos

In [28]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', None)

# Cargar el dataset
print("Buscando ruta del dataset...")
ruta_csv = 'global_concatenado/global_concatenado.CSV'
#si existe el archivo en la ruta especificada
try:    
    with open(ruta_csv, 'r') as file:
        print("Archivo encontrado.")
except FileNotFoundError:
    print(f"Error: El archivo '{ruta_csv}' no se encontró.")
    
print("Intentando cargar el dataset...")
try:
    df = pd.read_csv(ruta_csv)
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print(f"Error: El archivo '{ruta_csv}' no se encontró.")

print(df.head())
print(df.info())

Buscando ruta del dataset...
Archivo encontrado.
Intentando cargar el dataset...
Dataset cargado exitosamente.
   Unnamed: 0.1  Unnamed: 0 source  psi_psa1  psi_psa2  psi_psa3  psi_psa4  \
0             0           0  POXC1  61.16965  59.99485  66.90590  65.55704   
1             1           1  POXC1  61.72080  64.47652  62.18492  61.79332   
2             2           2  POXC1  66.52880  64.99865  66.77536  62.57652   
3             3           3  POXC1  62.67805  59.27691  66.49254  65.68032   
4             4           4  POXC1  60.76355  63.46850  61.97461  63.25094   

   psi_tablero   flujo  totalizador  ... suma_psa  suma_compresores  psi_psa5  \
0     49.44336  415.80    2234119.0  ...      4.0               4.0       NaN   
1     48.74717  405.84    2234125.0  ...      4.0               4.0       NaN   
2     49.78419  417.15    2234132.0  ...      4.0               4.0       NaN   
3     49.43610  416.88    2234138.0  ...      4.0               4.0       NaN   
4     40.52354 

In [29]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Tipo_de_Dato': df.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                       Nulos Tipo_de_Dato
Unnamed: 0.1               0        int64
Unnamed: 0                 0        int64
source                     0          str
psi_psa1              297228      float64
psi_psa2              297228      float64
psi_psa3              434287      float64
psi_psa4              434287      float64
psi_tablero           297228      float64
flujo                 280060      float64
totalizador           297228      float64
TIME                       0          str
r_psa1                  7649      float64
r_psa2                  7649      float64
r_psa3                144708      float64
r_psa4                144708      float64
r_gen1                 21204      float64
r_gen2                 21204      float64
r_bar                  21204      float64
r_sec1                 21204      float64
r_sec2                 21204      float64
r_com1                 21204      float64
r_com2                 21204      float64
r_com3                158263      


### **¿Qué significa cada columna? **

Sabiendo que la empresa se dedica a inyectar oxígeno a los peces las empresas utilizan plantas generadoras de oxígeno in situ.

Como tienes 109 columnas, la mejor forma de entenderlas es agruparlas por su **prefijo**, ya que siguen una nomenclatura estándar de telemetría industrial (sistemas SCADA o PLCs).

**1. Identificadores y Tiempo**

* `Unnamed: 0` / `Unnamed: 0.1`: Son índices antiguos o números de fila que se guardaron por error al exportar el CSV originalmente. (Te sugiero eliminarlas después).
* `source`: El origen de los datos o el identificador del pontón/centro de cultivo (ej. POXC1).
* `TIME`: La marca de tiempo (fecha y hora) exacta en la que se tomó la medición.
* `Sistema`: Probablemente el nombre o ID general del sistema operando.

**2. Sistema PSA (Generadores de Oxígeno)**
*Las plantas generan oxígeno separándolo del aire mediante un proceso llamado PSA (Pressure Swing Adsorption).*

* `psi_psa1` a `psi_psa6`: La presión (en PSI - libras por pulgada cuadrada) de cada uno de los generadores PSA (hasta 6 equipos).
* `suma_psa`: Cantidad total de módulos PSA que están encendidos o funcionando en ese momento.

**3. Flujo y Entrega de Oxígeno**

* `psi_tablero`: La presión de oxígeno en el tablero principal de distribución, justo antes de enviarlo a las jaulas de los peces.
* `flujo`: La cantidad de oxígeno que se está inyectando en ese instante (probablemente medido en litros por minuto o metros cúbicos por hora).
* `totalizador`: El volumen acumulado total de oxígeno que se ha entregado a lo largo del tiempo (como el cuentakilómetros de un auto).

**4. Variables de Estado o Funcionamiento (`r_`)**
*La "r" generalmente viene de "Run" (en marcha/corriendo) o "Relay". Indican si un equipo está encendido (1) o apagado (0).*

* `r_psa1` a `r_psa6`: Estado de marcha de los módulos PSA.
* `r_com1` a `r_com4`: Estado de marcha de los compresores de aire (el aire comprimido alimenta a los PSA).
* `suma_compresores`: Cantidad total de compresores encendidos.
* `r_gen1`, `r_gen2`: Estado de los generadores eléctricos.
* `r_sec1`, `r_sec2`: Estado de los secadores de aire (eliminan la humedad del aire antes de que entre al PSA).
* `r_bar`: Estado de la barredora o sistema de barrido.

**5. Variables de Control (`sp_`, `ox_`, `m_`)**
*Datos provenientes de los sensores instalados (probablemente en las distintas jaulas o líneas de inyección de oxígeno, numerados del s1 al s12).*

* `sp_s1` a `sp_s12`: **Setpoint** (Punto de ajuste). Es el nivel de oxígeno o presión objetivo que el operador programó en el sistema para ese sensor.
* `ox_s1` a `ox_s12`: La medición real de **concentración o nivel de oxígeno** (pureza) que está leyendo el sensor.
* `m_s1` a `m_s12`: Probablemente **Modo** de operación (ej. Automático vs Manual) o estado de la válvula/caudalímetro de ese sensor.

**6. Temperatura**

* `mb_g1_temperatura_f` / `mb_g2_temperatura_f`: La temperatura en grados Fahrenheit (°F) de los generadores o motores 1 y 2 (el prefijo "mb" suele referirse a Modbus, el protocolo de comunicación utilizado para extraer el dato).

**7. Señales del Sistema (`hb_`, `rst_`)**

* `hb_*` (ej. `hb_psa1`, `hb_com1`): **Heartbeat** (Latido). Es una señal digital (1 o 0) que el equipo envía cada segundo para decir "Estoy conectado y en línea". Si se pierde, significa que se cortó la comunicación de internet o red con el equipo.
* `rst_*` (ej. `rst_psa1`, `rst_com1`): **Reset / Restart**. Indica si se envió una señal de reinicio a ese equipo, o la cantidad de veces que se ha reiniciado por fallas.

# ATENCION !!!

no es noceario correr estas lineas buscar el estas si

In [30]:
# Condición: Que el valor NO (~) esté en la lista [0, 1] para alguna de las 3 columnas
condicion = (
    (~df['m_s1'].isin([0.0, 1.0])) | 
    (~df['m_s2'].isin([0.0, 1.0])) | 
    (~df['m_s3'].isin([0.0, 1.0]))
)

# Filtramos las columnas, quitamos los nulos
df_filtrado = df.loc[condicion, ['m_s1', 'm_s2', 'm_s3', 'ox_s1']].dropna()

# Vemos cuántos datos "raros" hay y mostramos los primeros 10
print(f"Se encontraron {len(df_filtrado)} filas con valores distintos de 0 y 1.\n")

if len(df_filtrado) > 0:
    print(df_filtrado.head(10))
else:
    print("¡Confirmado! Todos los datos en esas columnas son estrictamente 0.0 o 1.0")

Se encontraron 0 filas con valores distintos de 0 y 1.

¡Confirmado! Todos los datos en esas columnas son estrictamente 0.0 o 1.0


In [31]:
import pandas as pd

print("--- 1. Limpieza de las 3 columnas iniciales ---")
columnas_basura = ['Unnamed: 0.1', 'Unnamed: 0', 'Sistema']
df.drop(columns=[col for col in columnas_basura if col in df.columns], inplace=True)
print("Columnas eliminadas exitosamente.\n")

print("--- 2. Separación provisional (Creencias) ---")
# Creencia: Si tiene al menos un dato no nulo en r_com4, asumimos que tiene 4 compresores
sources_4_comp = df.dropna(subset=['r_com4'])['source'].unique()

df_4_creencia = df[df['source'].isin(sources_4_comp)]
df_3_creencia = df[~df['source'].isin(sources_4_comp)]

print(f"Filas en posible Sist. 4 Compresores: {len(df_4_creencia)}")
print(f"Filas en posible Sist. 3 Compresores: {len(df_3_creencia)}\n")

print("--- 3. Auditoría de las columnas (La prueba de fuego) ---")
columnas_psa_5_6 = [col for col in df.columns if 'psa5' in col or 'psa6' in col]
columnas_com4 = [col for col in df.columns if 'com4' in col]

# A. Revisión en el de 4 compresores (No debería tener PSA 5 ni 6)
total_filas_4 = len(df_4_creencia)
nulos_psa_5_6 = df_4_creencia[columnas_psa_5_6].isnull().sum()
cumple_4 = (nulos_psa_5_6 == total_filas_4).all()

print(f">> Sistema de 4 Compresores (Esperamos {total_filas_4} nulos en PSA 5 y 6):")
print(nulos_psa_5_6.to_string())
print(f"¿Cumple la regla de estar 100% vacías?: {'✅ SÍ' if cumple_4 else '❌ NO (hay datos intrusos)'}\n")

# B. Revisión en el de 3 compresores (No debería tener Compresor 4)
total_filas_3 = len(df_3_creencia)
nulos_com4 = df_3_creencia[columnas_com4].isnull().sum()
cumple_3 = (nulos_com4 == total_filas_3).all()

print(f">> Sistema de 3 Compresores (Esperamos {total_filas_3} nulos en Compresor 4):")
print(nulos_com4.to_string())
print(f"¿Cumple la regla de estar 100% vacías?: {'✅ SÍ' if cumple_3 else '❌ NO (hay datos intrusos)'}\n")

# Veredicto final
if cumple_4 and cumple_3:
    print("🎯 ¡Todo nice! Las creencias son correctas. Listos para dividir oficialmente.")
else:
    print("⚠️ Ojo: Las reglas no se cumplieron al 100%. Tenemos datos donde no deberían estar.")

--- 1. Limpieza de las 3 columnas iniciales ---
Columnas eliminadas exitosamente.

--- 2. Separación provisional (Creencias) ---
Filas en posible Sist. 4 Compresores: 5862409
Filas en posible Sist. 3 Compresores: 2781783

--- 3. Auditoría de las columnas (La prueba de fuego) ---
>> Sistema de 4 Compresores (Esperamos 5862409 nulos en PSA 5 y 6):
psi_psa5    5862409
psi_psa6    5862409
r_psa5      5862409
r_psa6      5862409
hb_psa5     5862409
hb_psa6     5862409
rst_psa5    5862409
rst_psa6    5862409
¿Cumple la regla de estar 100% vacías?: ✅ SÍ

>> Sistema de 3 Compresores (Esperamos 2781783 nulos en Compresor 4):
r_com4      2781783
hb_com4     2133884
rst_com4    2133884
¿Cumple la regla de estar 100% vacías?: ❌ NO (hay datos intrusos)

⚠️ Ojo: Las reglas no se cumplieron al 100%. Tenemos datos donde no deberían estar.


In [32]:
print("--- Verificación de señales fantasma en Sistema de 3 Compresores ---")

# Obtenemos los valores únicos de hb_com4 y rst_com4, ignorando los nulos (NaN)
valores_hb = df_3_creencia['hb_com4'].dropna().unique()
valores_rst = df_3_creencia['rst_com4'].dropna().unique()

print(f"Valores reales encontrados en hb_com4: {valores_hb}")
print(f"Valores reales encontrados en rst_com4: {valores_rst}\n")

# Hacemos una pequeña validación automática
if len(valores_hb) == 1 and valores_hb[0] == 0.0 and len(valores_rst) == 1 and valores_rst[0] == 0.0:
    print("✅ Todo en orden. Los sensores solo arrojan '0.0'. Es un equipo fantasma.")
    print("Puedes proceder con la purga con total seguridad.")
else:
    print("⚠️ ¡ALTO AHI! Hay valores distintos de 0. Podría haber un problema de mapeo en el PLC.")

--- Verificación de señales fantasma en Sistema de 3 Compresores ---
Valores reales encontrados en hb_com4: [1.]
Valores reales encontrados en rst_com4: [0.]

⚠️ ¡ALTO AHI! Hay valores distintos de 0. Podría haber un problema de mapeo en el PLC.


In [33]:
# Seleccionamos las columnas de interés para tener contexto visual
columnas_ver = ['source', 'TIME', 'r_com3', 'r_com4', 'hb_com4', 'rst_com4','psi_psa1','psi_psa2','psi_psa3','psi_psa4','psi_psa5', 'psi_psa6', 'r_psa5', 'r_psa6', 'hb_psa5', 'hb_psa6', 'rst_psa5', 'rst_psa6']

# Filtramos las filas en el sistema de 3 compresores donde hb_com4 tiene un latido (1.0)
df_fantasmas = df_3_creencia[df_3_creencia['hb_com4'] == 1.0][columnas_ver]

print("Primeras 5 filas con el 'latido fantasma' (hb_com4 = 1.0):")
print(df_fantasmas.head(5))

df_fantasmas_rst = df_3_creencia[df_3_creencia['rst_com4'].notnull()][columnas_ver]
print("\nPrimeras 5 filas con el 'reset fantasma' (rst_com4 = 1.0):")
print(df_fantasmas_rst.head(5))

Primeras 5 filas con el 'latido fantasma' (hb_com4 = 1.0):
      source                     TIME  r_com3  r_com4  hb_com4  rst_com4  \
42115  POX47  2026-02-01 00:00:20.190     0.0     NaN      1.0       0.0   
42116  POX47  2026-02-01 00:01:20.191     0.0     NaN      1.0       0.0   
42117  POX47  2026-02-01 00:02:20.192     0.0     NaN      1.0       0.0   
42118  POX47  2026-02-01 00:03:20.193     0.0     NaN      1.0       0.0   
42119  POX47  2026-02-01 00:04:20.194     0.0     NaN      1.0       0.0   

       psi_psa1  psi_psa2  psi_psa3  psi_psa4  psi_psa5  psi_psa6  r_psa5  \
42115 -0.007251 -0.181296  0.029009 -0.130533       NaN       NaN     NaN   
42116 -0.007251 -0.181296  0.029009 -0.130533       NaN       NaN     NaN   
42117 -0.007251 -0.181296  0.029009 -0.130533       NaN       NaN     NaN   
42118 -0.007251 -0.181296  0.029009 -0.130533       NaN       NaN     NaN   
42119 -0.007251 -0.174045  0.029009 -0.130533       NaN       NaN     NaN   

       r_psa6  hb_psa

en teoria no debirea existr 3 comresores a 4 psa (segun el experto), asi q vamos a revisar si el sours es el fallado o hay mas plantas que ocupen esa distribucion

In [34]:
import pandas as pd

print("--- Auditoría de Arquitecturas por Source (Sistemas de 3 Compresores) ---")

# 1. Agrupamos por 'source' y contamos el total de filas y los nulos en columnas clave
resumen_sources = df_3_creencia.groupby('source').agg(
    total_filas=('source', 'count'),
    nulos_r_com4=('r_com4', lambda x: x.isnull().sum()),
    nulos_psa5=('psi_psa5', lambda x: x.isnull().sum()),
    nulos_hb_com4=('hb_com4', lambda x: x.isnull().sum())
)

# 2. Buscamos los que son de 3 Compresores (r_com4 vacío) y 4 PSA (psa5 vacío)
fuentes_3comp_4psa = resumen_sources[
    (resumen_sources['nulos_r_com4'] == resumen_sources['total_filas']) & 
    (resumen_sources['nulos_psa5'] == resumen_sources['total_filas'])
]

print("\n1. Centros que tienen 3 Compresores y SOLO 4 PSA:")
print(fuentes_3comp_4psa[['total_filas', 'nulos_r_com4', 'nulos_psa5']])

# 3. Verificamos si en todos ellos existe el "fantasma" de hb_com4
print("\n2. Revisando el latido fantasma en estos centros...")
fantasmas_confirmados = fuentes_3comp_4psa[fuentes_3comp_4psa['nulos_hb_com4'] < fuentes_3comp_4psa['total_filas']]

if len(fantasmas_confirmados) == len(fuentes_3comp_4psa):
    print("✅ ¡Confirmado! TODOS los centros con esta arquitectura tienen el latido fantasma en hb_com4.")
    print(f"Lista de sources: {fantasmas_confirmados.index.tolist()}")
else:
    print("⚠️ Ojo: Solo algunos tienen el fantasma. Aquí está el detalle:")
    print(fantasmas_confirmados[['total_filas', 'nulos_hb_com4']])

--- Auditoría de Arquitecturas por Source (Sistemas de 3 Compresores) ---

1. Centros que tienen 3 Compresores y SOLO 4 PSA:
        total_filas  nulos_r_com4  nulos_psa5
source                                       
POX1         137539        137539      137539
POX47        157314        157314      157314
POX64        163709        163709      163709
POX66        163612        163612      163612
POX67        163738        163738      163738

2. Revisando el latido fantasma en estos centros...
⚠️ Ojo: Solo algunos tienen el fantasma. Aquí está el detalle:
        total_filas  nulos_hb_com4
source                            
POX47        157314            100
POX64        163709              2
POX66        163612              0
POX67        163738            372


POX66: Tiene 163,612 filas. ¿Cuántos nulos tiene en hb_com4? Cero (0). Esto significa que durante 163,612 minutos (o segundos), el sistema estuvo enviando una señal fantasma diciendo "¡El compresor 4 está vivo!".

POX64: Tiene 163,709 filas y solo 2 nulos. Es decir, la señal fantasma estuvo transmitiendo todo el tiempo y solo se cayó o apagó 2 veces.

¿Y dónde está POX1? ¡POX1 no aparece en esta lista de alerta! Eso es una excelente noticia. Significa que en POX1, de sus 137,539 filas, hay 137,539 nulos. POX1 es el único pontón bien programado que no manda datos basura del motor que le falta.

In [35]:
import pandas as pd

print("--- Eliminación de Pontones Anómalos ---")

# Lista de pontones con arquitectura irreal (3 compresores y 4 PSA)
pontones_basura = ['POX1', 'POX47', 'POX64', 'POX66', 'POX67']

# Filtramos el dataset para quedarnos con todo lo que NO esté en esa lista
df_filtrado = df[~df['source'].isin(pontones_basura)].copy()

# Vemos cuánto peso muerto acabamos de soltar
filas_eliminadas = len(df) - len(df_filtrado)
print(f"Total de filas originales: {len(df)}")
print(f"Pontones anómalos eliminados exitosamente.")
print(f"Filas eliminadas (basura): {filas_eliminadas}")
print(f"Total de filas limpias: {len(df_filtrado)}\n")

--- Eliminación de Pontones Anómalos ---
Total de filas originales: 8644192
Pontones anómalos eliminados exitosamente.
Filas eliminadas (basura): 785912
Total de filas limpias: 7858280



# ESTAS SI

In [36]:
import pandas as pd
import gc  # Librería nativa para limpiar la memoria RAM

print("--- 1. Salvando la Memoria RAM (Limpieza In-Place) ---")

columnas_basura = ['Unnamed: 0.1', 'Unnamed: 0', 'Sistema']
df.drop(columns=[col for col in columnas_basura if col in df.columns], inplace=True)
print("✅ Columnas iniciales eliminadas.")

pontones_basura = ['POX1', 'POX47', 'POX64', 'POX66', 'POX67']
indices_basura = df[df['source'].isin(pontones_basura)].index
df.drop(index=indices_basura, inplace=True)
print(f"✅ Pontones anómalos eliminados ({len(indices_basura)} filas).")

print("\n--- 2. Compresión Extrema de Datos (Lógica Simplificada) ---")
print("⏳ Comprimiendo columnas... (Esto tomará unos minutos)")

for col in df.columns:
    if df[col].dtype == 'float64':
        # Variables que ESPERAMOS que sean 0 o 1 (r_, m_, hb_, rst_, suma_)
        if col.startswith(('r_', 'm_', 'hb_', 'rst_')) or col == 'suma_compresores':
            
            min_val = df[col].min()
            max_val = df[col].max()
            
            # Si solo tiene nulos, o sus valores están entre -128 y 127 (cubre 0 y 1)
            if pd.isna(max_val) or (min_val >= -128 and max_val <= 127):
                df[col] = df[col].astype('Int8')
            else:
                # Si es un número gigante o raro, lo dejamos como float (pero más liviano)
                df[col] = df[col].astype('float32')
                
        # Para el resto de las métricas (presión, flujo, oxígeno)
        else:
            df[col] = df[col].astype('float32')

gc.collect()
print("✅ Compresión completada. ¡Tu RAM está a salvo!")

print("\n--- 3. Separación Oficial ---")
sources_4_comp = df.dropna(subset=['r_com4'])['source'].unique()

df_4comp = df[df['source'].isin(sources_4_comp)].copy()
df_3comp = df[~df['source'].isin(sources_4_comp)].copy()

# Eliminamos el dataframe original gigante
del df
gc.collect()

print(f"Filas en Sist. 4 Compresores (4 PSA): {len(df_4comp)}")
print(f"Filas en Sist. 3 Compresores (6 PSA): {len(df_3comp)}\n")

print("--- 4. Purga Final ---")
columnas_psa_5_6 = [col for col in df_4comp.columns if 'psa5' in col or 'psa6' in col]
columnas_com4 = [col for col in df_3comp.columns if 'com4' in col]

df_4comp.drop(columns=columnas_psa_5_6, inplace=True, errors='ignore')
df_3comp.drop(columns=columnas_com4, inplace=True, errors='ignore')

print("🚀 ¡Proceso completado exitosamente!")
print(f"Dataframe 4 Compresores: {df_4comp.shape[1]} columnas")
print(f"Dataframe 3 Compresores: {df_3comp.shape[1]} columnas")

--- 1. Salvando la Memoria RAM (Limpieza In-Place) ---
✅ Columnas iniciales eliminadas.
✅ Pontones anómalos eliminados (785912 filas).

--- 2. Compresión Extrema de Datos (Lógica Simplificada) ---
⏳ Comprimiendo columnas... (Esto tomará unos minutos)
✅ Compresión completada. ¡Tu RAM está a salvo!

--- 3. Separación Oficial ---
Filas en Sist. 4 Compresores (4 PSA): 5862409
Filas en Sist. 3 Compresores (6 PSA): 1995871

--- 4. Purga Final ---
🚀 ¡Proceso completado exitosamente!
Dataframe 4 Compresores: 98 columnas
Dataframe 3 Compresores: 103 columnas


In [37]:
print("\nMostrando las primeras filas del DataFrame de 4 Compresores:"      )
print(df_4comp.head())
print("\nMostrando las primeras filas del DataFrame de 3 Compresores:"      )
print(df_3comp.head())


Mostrando las primeras filas del DataFrame de 4 Compresores:
  source   psi_psa1   psi_psa2   psi_psa3   psi_psa4  psi_tablero       flujo  \
0  POXC1  61.169651  59.994850  66.905899  65.557037    49.443359  415.799988   
1  POXC1  61.720798  64.476517  62.184921  61.793320    48.747169  405.839996   
2  POXC1  66.528801  64.998650  66.775360  62.576519    49.784191  417.149994   
3  POXC1  62.678051  59.276909  66.492538  65.680321    49.436100  416.880005   
4  POXC1  60.763550  63.468498  61.974609  63.250938    40.523540  404.940002   

   totalizador                     TIME  r_psa1  ...  m_s7  m_s8  m_s9  m_s10  \
0    2234119.0  2026-02-01 00:00:11.078       1  ...     0     0     0      0   
1    2234125.0  2026-02-01 00:01:11.079       1  ...     0     0     0      0   
2    2234132.0  2026-02-01 00:02:11.080       1  ...     0     0     0      0   
3    2234138.0  2026-02-01 00:03:11.081       1  ...     0     0     0      0   
4    2234144.0  2026-02-01 00:04:11.082       

In [38]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_4comp.isnull().sum(),
    'Tipo_de_Dato': df_4comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())


                      Nulos Tipo_de_Dato
source                    0          str
psi_psa1             292524      float32
psi_psa2             292524      float32
psi_psa3             292524      float32
psi_psa4             292524      float32
psi_tablero          292524      float32
flujo                278615      float32
totalizador          292524      float32
TIME                      0          str
r_psa1                 4557         Int8
r_psa2                 4557         Int8
r_psa3                 4557         Int8
r_psa4                 4557         Int8
r_gen1                16500         Int8
r_gen2                16500         Int8
r_bar                 16500         Int8
r_sec1                16500         Int8
r_sec2                16500         Int8
r_com1                16500         Int8
r_com2                16500         Int8
r_com3                16500         Int8
r_com4                16500         Int8
sp_s1                327409      float32
sp_s2           

In [39]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_3comp.isnull().sum(),
    'Tipo_de_Dato': df_3comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                     Nulos Tipo_de_Dato
source                   0          str
psi_psa1              3440      float32
psi_psa2              3440      float32
psi_psa3              3440      float32
psi_psa4              3440      float32
psi_tablero           3440      float32
flujo                  962      float32
totalizador           3440      float32
TIME                     0          str
r_psa1                2609         Int8
r_psa2                2609         Int8
r_psa3                2609         Int8
r_psa4                2609         Int8
r_gen1                3440         Int8
r_gen2                3440         Int8
r_bar                 3440         Int8
r_sec1                3440         Int8
r_sec2                3440         Int8
r_com1                3440         Int8
r_com2                3440         Int8
r_com3                3440         Int8
sp_s1                10261      float32
sp_s2                10261      float32
sp_s3                10261      float32


In [40]:
# Ver cuántos datos válidos (no nulos) tiene cada jaula por cada pontón
# Esto te dirá exactamente cuántas jaulas físicas tiene cada 'source'
jaulas_por_ponton = df_4comp.groupby('source')[['ox_s1', 'ox_s8', 'ox_s9', 'ox_s12']].count()
print(jaulas_por_ponton)

         ox_s1   ox_s8   ox_s9  ox_s12
source                                
POX10   161478  161478  161478  161478
POX11    43080   43080   43080   43080
POX13    93824   93824   93824   93824
POX15   160124  160124       0  160504
POX16   158380  158380  158380  158380
POX17   159223  159223  159223  159223
POX18   158178  158178  158178  158178
POX19    44276   44276   44276   44276
POX20   163071  163071  163071  163071
POX21   162894  162894  162894  162894
POX22   162735  162735  162735  162735
POX23   121844  121844  121844  121844
POX24   162627  162627       0  162147
POX25   102812  102812  102812  102812
POX26   163225  163225  163225  163225
POX27   128549  128549       0  128649
POX28   163162  163162  163162  163162
POX29   163092  163092  163092  163092
POX3     21402   21402       0   22561
POX30   138339  138339  138339  138339
POX31   163183  163183  163183  163183
POX32   132768  132768  132868  132868
POX33   151302  151302  156871  156871
POX34   108229  108229  1

## por que el sensor 1 falla?

In [41]:
import pandas as pd

print("--- DIAGNÓSTICO DE NULOS EN SENSORES DE OXÍGENO ---")

# Seleccionamos las 12 columnas de oxígeno
cols_ox = [f'ox_s{i}' for i in range(1, 13)]

def analizar_nulos_sensores(df, nombre_sistema):
    print(f"\n>> Analizando: {nombre_sistema}")
    
    # 1. Total de nulos por cada sensor
    nulos_por_sensor = df[cols_ox].isnull().sum()
    print("Nulos individuales por sensor:")
    print(nulos_por_sensor.to_string())
    
    # 2. Análisis de Apagones (Blackouts)
    # ¿Cuántas filas tienen TODOS los sensores en nulo al mismo tiempo?
    apagones_totales = df[cols_ox].isnull().all(axis=1).sum()
    porcentaje_apagones = (apagones_totales / len(df)) * 100
    
    print(f"\nFilas con APAGÓN TOTAL (Todos los sensores nulos a la vez): {apagones_totales} ({porcentaje_apagones:.2f}%)")
    
    # 3. Ver qué pontones causan los nulos en el sensor 1
    # Filtramos las filas donde ox_s1 es nulo, pero NO es un apagón total
    fallas_aisladas_s1 = df[df['ox_s1'].isnull() & ~df[cols_ox].isnull().all(axis=1)]
    if len(fallas_aisladas_s1) > 0:
        print("\nPontones con fallas AISLADAS en el Sensor 1:")
        print(fallas_aisladas_s1['source'].value_counts())
    else:
        print("\nTodos los nulos del Sensor 1 ocurren durante apagones generales.")

# Ejecutamos para ambos datasets
analizar_nulos_sensores(df_4comp, "Sistema de 4 Compresores")
analizar_nulos_sensores(df_3comp, "Sistema de 3 Compresores")

--- DIAGNÓSTICO DE NULOS EN SENSORES DE OXÍGENO ---

>> Analizando: Sistema de 4 Compresores
Nulos individuales por sensor:
ox_s1     320123
ox_s2     320123
ox_s3     320123
ox_s4     320123
ox_s5     320123
ox_s6     320123
ox_s7     320123
ox_s8     320123
ox_s9     787156
ox_s10    313295
ox_s11    313295
ox_s12    313295

Filas con APAGÓN TOTAL (Todos los sensores nulos a la vez): 312709 (5.33%)

Pontones con fallas AISLADAS en el Sensor 1:
source
POX33    5571
POX3     1160
POX15     482
POX27     101
POX32     100
Name: count, dtype: int64

>> Analizando: Sistema de 3 Compresores
Nulos individuales por sensor:
ox_s1     8645
ox_s2     8645
ox_s3     8645
ox_s4     8645
ox_s5     8645
ox_s6     8645
ox_s7     8645
ox_s8     8645
ox_s9     8645
ox_s10    8645
ox_s11    8645
ox_s12    8645

Filas con APAGÓN TOTAL (Todos los sensores nulos a la vez): 8645 (0.43%)

Todos los nulos del Sensor 1 ocurren durante apagones generales.


In [42]:
import pandas as pd

print("--- ANÁLISIS TEMPORAL DE APAGONES (BLACKOUTS) ---")
cols_ox = [f'ox_s{i}' for i in range(1, 13)]

def analizar_fechas_apagones(df, nombre_sistema):
    print(f"\n>> Top 10 días con más apagones en: {nombre_sistema}")
    
    # 1. Filtramos solo las filas que son apagones totales
    apagones = df[df[cols_ox].isnull().all(axis=1)].copy()
    
    if len(apagones) == 0:
        print("No hay apagones en este sistema.")
        return
        
    # 2. Convertimos la columna TIME a tipo datetime y extraemos solo la fecha (YYYY-MM-DD)
    # Usamos errors='coerce' por si hay algún texto raro infiltrado en las fechas
    apagones['Fecha'] = pd.to_datetime(apagones['TIME'], errors='coerce').dt.date
    
    # 3. Contamos cuántos registros de apagón cayeron en cada día
    conteo_dias = apagones['Fecha'].value_counts().head(10)
    
    print(conteo_dias.to_string())
    print(f"(Total de apagones analizados: {len(apagones)})")

# Ejecutamos la función
analizar_fechas_apagones(df_4comp, "Sistema de 4 Compresores")
analizar_fechas_apagones(df_3comp, "Sistema de 3 Compresores")

--- ANÁLISIS TEMPORAL DE APAGONES (BLACKOUTS) ---

>> Top 10 días con más apagones en: Sistema de 4 Compresores
Fecha
2026-03-06    6170
2026-02-16    5779
2026-02-26    5761
2026-03-01    5761
2026-02-14    5760
2026-02-20    5760
2026-02-28    5759
2026-02-03    5746
2026-02-24    5697
2026-02-13    5525
(Total de apagones analizados: 312709)

>> Top 10 días con más apagones en: Sistema de 3 Compresores
Fecha
2026-05-20    1440
2026-05-21    1440
2026-05-22    1440
2026-05-23    1065
2026-03-06     960
2026-05-19     583
2026-03-11     481
2026-03-12     480
2026-05-24     480
2026-04-02     200
(Total de apagones analizados: 8645)


In [43]:
import pandas as pd

print("--- ANÁLISIS DE APAGONES: ¿FALLA REGIONAL O AISLADA? ---")
cols_ox = [f'ox_s{i}' for i in range(1, 13)]

def analizar_apagones_por_ponton(df, nombre_sistema):
    print(f"\n>> Detalle de apagones en: {nombre_sistema}")
    
    # 1. Filtramos los apagones
    apagones = df[df[cols_ox].isnull().all(axis=1)].copy()
    
    if len(apagones) == 0:
        print("No hay apagones en este sistema.")
        return
        
    # 2. Extraemos la fecha
    apagones['Fecha'] = pd.to_datetime(apagones['TIME'], errors='coerce').dt.date
    
    # 3. Agrupamos por Fecha y por Pontón (source)
    detalle = apagones.groupby(['Fecha', 'source']).size().reset_index(name='minutos_caidos')
    
    # 4. Ordenamos para ver los peores casos arriba y mostramos el Top 15
    detalle = detalle.sort_values(by='minutos_caidos', ascending=False).head(15)
    
    print(detalle.to_string(index=False))

# Ejecutamos la función
analizar_apagones_por_ponton(df_4comp, "Sistema de 4 Compresores")
analizar_apagones_por_ponton(df_3comp, "Sistema de 3 Compresores")

--- ANÁLISIS DE APAGONES: ¿FALLA REGIONAL O AISLADA? ---

>> Detalle de apagones en: Sistema de 4 Compresores
     Fecha source  minutos_caidos
2026-04-13  POX19            1440
2026-04-12  POX19            1440
2026-04-12  POX11            1440
2026-04-11  POX19            1440
2026-04-10  POX11            1440
2026-04-09  POX19            1440
2026-04-09  POX11            1440
2026-04-08  POX19            1440
2026-04-08  POX11            1440
2026-02-08  POX35            1440
2026-02-08  POX19            1440
2026-02-07  POX35            1440
2026-02-07  POX19            1440
2026-02-06  POX35            1440
2026-02-06  POX19            1440

>> Detalle de apagones en: Sistema de 3 Compresores
     Fecha source  minutos_caidos
2026-05-22  POX42            1440
2026-05-20  POX42            1440
2026-05-21  POX42            1440
2026-05-23  POX42            1065
2026-05-19  POX42             582
2026-03-06  POX59             480
2026-03-06  POX40             480
2026-03-11  POX40    

## como son apagones por un dia entero o por manteciones programadas:
antes de eliminar las columnas vamos a guardar esa info:

In [44]:
import pandas as pd
import numpy as np

print("--- INGENIERÍA DE CARACTERÍSTICAS: HISTORIAL DE MANTENIMIENTO ---")
cols_ox = [f'ox_s{i}' for i in range(1, 13)]

def crear_memoria_temporal(df, nombre_sistema):
    print(f"\n⏳ Procesando {nombre_sistema}...")
    
    # 1. Asegurarnos de que TIME sea formato de fecha real y ordenar cronológicamente
    df['TIME'] = pd.to_datetime(df['TIME'], errors='coerce')
    df.sort_values(by=['source', 'TIME'], inplace=True)
    
    # 2. Identificar qué filas son apagones totales
    df['es_apagon'] = df[cols_ox].isnull().all(axis=1)
    
    # 3. Guardar el timestamp SOLO en las filas donde hay un apagón
    df.loc[df['es_apagon'], 'tiempo_apagon'] = df['TIME']
    
    # 4. Magia de Pandas: Arrastrar ese tiempo hacia adelante (por pontón)
    df['ultimo_apagon'] = df.groupby('source')['tiempo_apagon'].ffill()
    
    # 5. Calcular la diferencia en horas
    # Si 'ultimo_apagon' es NaT (nunca ha tenido uno), el resultado será NaN
    diferencia_segundos = (df['TIME'] - df['ultimo_apagon']).dt.total_seconds()
    df['horas_desde_mantencion'] = diferencia_segundos / 3600.0
    
    # 6. Rellenar los que nunca han tenido apagones con un número muy grande (9999)
    df['horas_desde_mantencion'] = df['horas_desde_mantencion'].fillna(9999.0).astype('float32')
    
    # --- AHORA SÍ PODEMOS LIMPIAR ---
    filas_antes = len(df)
    
    # 7. Borrar las filas que son apagones (ya no las necesitamos)
    df.drop(df[df['es_apagon']].index, inplace=True)
    
    # 8. Rellenar la falta física de jaulas con -1
    df[cols_ox] = df[cols_ox].fillna(-1)
    
    # 9. Limpiar columnas temporales de cálculo
    df.drop(columns=['es_apagon', 'tiempo_apagon', 'ultimo_apagon'], inplace=True)
    
    print(f"✅ Variable 'horas_desde_mantencion' creada exitosamente.")
    print(f"✅ Apagones eliminados: {filas_antes - len(df)} filas.")
    print(f"✅ Jaulas faltantes rellenadas con -1.")
    
    return df

# Aplicamos la función a ambos datasets
df_4comp = crear_memoria_temporal(df_4comp, "Sistema de 4 Compresores")
df_3comp = crear_memoria_temporal(df_3comp, "Sistema de 3 Compresores")

--- INGENIERÍA DE CARACTERÍSTICAS: HISTORIAL DE MANTENIMIENTO ---

⏳ Procesando Sistema de 4 Compresores...
✅ Variable 'horas_desde_mantencion' creada exitosamente.
✅ Apagones eliminados: 312709 filas.
✅ Jaulas faltantes rellenadas con -1.

⏳ Procesando Sistema de 3 Compresores...
✅ Variable 'horas_desde_mantencion' creada exitosamente.
✅ Apagones eliminados: 8645 filas.
✅ Jaulas faltantes rellenadas con -1.


In [45]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_4comp.isnull().sum(),
    'Tipo_de_Dato': df_4comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                        Nulos    Tipo_de_Dato
source                      0             str
psi_psa1                12178         float32
psi_psa2                12178         float32
psi_psa3                12178         float32
psi_psa4                12178         float32
psi_tablero             12178         float32
flujo                    1261         float32
totalizador             12178         float32
TIME                        0  datetime64[us]
r_psa1                   2745            Int8
r_psa2                   2745            Int8
r_psa3                   2745            Int8
r_psa4                   2745            Int8
r_gen1                  12177            Int8
r_gen2                  12177            Int8
r_bar                   12177            Int8
r_sec1                  12177            Int8
r_sec2                  12177            Int8
r_com1                  12177            Int8
r_com2                  12177            Int8
r_com3                  12177     

In [46]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_3comp.isnull().sum(),
    'Tipo_de_Dato': df_3comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                        Nulos    Tipo_de_Dato
source                      0             str
psi_psa1                 1508         float32
psi_psa2                 1508         float32
psi_psa3                 1508         float32
psi_psa4                 1508         float32
psi_tablero              1508         float32
flujo                       0         float32
totalizador              1508         float32
TIME                        0  datetime64[us]
r_psa1                    681            Int8
r_psa2                    681            Int8
r_psa3                    681            Int8
r_psa4                    681            Int8
r_gen1                   1508            Int8
r_gen2                   1508            Int8
r_bar                    1508            Int8
r_sec1                   1508            Int8
r_sec2                   1508            Int8
r_com1                   1508            Int8
r_com2                   1508            Int8
r_com3                   1508     

In [47]:
import pandas as pd

print("--- INSPECCIÓN DE NULOS EN PRESIONES (psi_psa1) ---")

# Seleccionamos columnas clave de la máquina y un par de jaulas para dar contexto
columnas_ver = ['source', 'TIME', 'psi_psa1', 'psi_psa2', 'r_com1', 'r_psa1', 'ox_s1', 'ox_s8']

# Filtramos las filas donde psi_psa1 es nulo
df_psi_nulo = df_4comp[df_4comp['psi_psa1'].isnull()][columnas_ver]

print(f"Total de filas con psi_psa1 nulo: {len(df_psi_nulo)}\n")

print(">> Primeras 15 filas con este problema:")
print(df_psi_nulo.head(15).to_string())

print("\n>> ¿Qué pontones concentran estos nulos?")
# Vemos si están repartidos o si es culpa de un solo pontón problemático
print(df_psi_nulo['source'].value_counts().head(10).to_string())

--- INSPECCIÓN DE NULOS EN PRESIONES (psi_psa1) ---
Total de filas con psi_psa1 nulo: 12178

>> Primeras 15 filas con este problema:
        source                    TIME  psi_psa1  psi_psa2  r_com1  r_psa1     ox_s1  ox_s8
4503778  POX10 2026-04-02 01:10:49.426       NaN       NaN    <NA>    <NA>  7.331290    0.0
4503779  POX10 2026-04-02 01:11:49.427       NaN       NaN    <NA>    <NA>  7.192930    0.0
4503780  POX10 2026-04-02 01:12:49.429       NaN       NaN    <NA>    <NA>  7.210235    0.0
4503781  POX10 2026-04-02 01:13:49.430       NaN       NaN    <NA>    <NA>  7.053269    0.0
4503782  POX10 2026-04-02 01:14:49.432       NaN       NaN    <NA>    <NA>  7.005767    0.0
4503783  POX10 2026-04-02 01:15:49.433       NaN       NaN    <NA>    <NA>  7.422217    0.0
4503784  POX10 2026-04-02 01:16:49.435       NaN       NaN    <NA>    <NA>  7.487130    0.0
4503785  POX10 2026-04-02 01:17:49.436       NaN       NaN    <NA>    <NA>  7.267754    0.0
4503786  POX10 2026-04-02 01:18:49.438 

In [48]:
import pandas as pd

print("--- ANÁLISIS DE DURACIÓN DE CORTES EN SALA DE MÁQUINAS ---")

def analizar_duracion_cortes(df, nombre_sistema, columna='psi_psa1'):
    print(f"\n>> Analizando {nombre_sistema} (Sensor: {columna})")
    
    # Verificar si hay nulos
    if df[columna].isnull().sum() == 0:
        print("No hay cortes en este sensor.")
        return
        
    # Asegurar orden cronológico por pontón
    df.sort_values(by=['source', 'TIME'], inplace=True)
    
    # Crear una serie booleana donde True es un nulo
    es_nulo = df[columna].isnull()
    
    # TRUCO MÁGICO: Crear un ID único para cada "bloque" continuo de nulos
    id_bloques = (es_nulo != es_nulo.shift()).cumsum()
    
    # Agrupar solo las filas que SON nulos por su ID de bloque y contar cuántas filas tiene cada uno
    # Como tenemos un dato por minuto, el tamaño del bloque = minutos caídos
    duracion_cortes = df[es_nulo].groupby(id_bloques).size()
    
    promedio = duracion_cortes.mean()
    mediana = duracion_cortes.median()
    maximo = duracion_cortes.max()
    
    print(f"Total de cortes (eventos de desconexión): {len(duracion_cortes)}")
    print(f"Duración PROMEDIO del corte: {promedio:.1f} minutos")
    print(f"Duración MEDIANA del corte: {mediana:.1f} minutos")
    print(f"Duración MÁXIMA (el peor caso): {maximo} minutos")
    
    print("\nTop 5 cortes más largos (minutos):")
    # Imprimimos los valores más altos
    print(duracion_cortes.sort_values(ascending=False).head(5).to_string(index=False))

# Aplicamos la función usando psi_psa1 como referencia de la sala de máquinas
analizar_duracion_cortes(df_4comp, "Sistema de 4 Compresores")
analizar_duracion_cortes(df_3comp, "Sistema de 3 Compresores")

--- ANÁLISIS DE DURACIÓN DE CORTES EN SALA DE MÁQUINAS ---

>> Analizando Sistema de 4 Compresores (Sensor: psi_psa1)
Total de cortes (eventos de desconexión): 46
Duración PROMEDIO del corte: 264.7 minutos
Duración MEDIANA del corte: 370.5 minutos
Duración MÁXIMA (el peor caso): 655 minutos

Top 5 cortes más largos (minutos):
psi_psa1
655
480
480
480
480

>> Analizando Sistema de 3 Compresores (Sensor: psi_psa1)
Total de cortes (eventos de desconexión): 9
Duración PROMEDIO del corte: 167.6 minutos
Duración MEDIANA del corte: 100.0 minutos
Duración MÁXIMA (el peor caso): 480 minutos

Top 5 cortes más largos (minutos):
psi_psa1
480
480
344
100
100


## como hay CORTES EN SALA DE MÁQUINAS pero las maquinas si funcionan

Confianza a corto plazo: Si el sensor se cae por 15 minutos o menos, asumimos que la máquina siguió haciendo lo mismo. (Usaremos un ffill con un límite).

Desconexión a largo plazo: Si pasa de 15 minutos, ya no confiamos. Lo rellenamos con nuestro salvavidas: el -1. Así el Random Forest entenderá que "durante estas 8 horas, la sala de máquinas estuvo desconectada del sistema".

In [49]:
print("--- LIMPIEZA FINAL: PROTECCIÓN CONTRA CORTES LARGOS ---")

def reparar_telemetria_inteligente(df, nombre_sistema):
    print(f"\n⏳ Reparando sensores en {nombre_sistema}...")
    
    # 1. Aseguramos el orden cronológico
    df.sort_values(by=['source', 'TIME'], inplace=True)
    
    # 2. Identificamos las variables de la máquina
    cols_ox = [f'ox_s{i}' for i in range(1, 13)]
    cols_excluir = ['source', 'TIME', 'horas_desde_mantencion'] + cols_ox
    cols_maquina = df.columns.drop(cols_excluir)
    
    # 3. Relleno corto: Máximo 15 minutos (limit=15)
    # Rellenamos hacia adelante, pero si el hueco es mayor a 15, dejará los demás como nulos
    df[cols_maquina] = df.groupby('source')[cols_maquina].ffill(limit=15)
    
    # 4. Relleno largo: Todo lo que sobró (cortes mayores a 15 min), lo marcamos como -1
    df[cols_maquina] = df[cols_maquina].fillna(-1)
    
    # Validación final
    nulos_restantes = df.isnull().sum().sum()
    print(f"✅ Reparación completa. Nulos totales en el dataset: {nulos_restantes}")
    
    return df

# Aplicamos a ambos
df_4comp = reparar_telemetria_inteligente(df_4comp, "Sistema de 4 Compresores")
df_3comp = reparar_telemetria_inteligente(df_3comp, "Sistema de 3 Compresores")

--- LIMPIEZA FINAL: PROTECCIÓN CONTRA CORTES LARGOS ---

⏳ Reparando sensores en Sistema de 4 Compresores...
✅ Reparación completa. Nulos totales en el dataset: 0

⏳ Reparando sensores en Sistema de 3 Compresores...
✅ Reparación completa. Nulos totales en el dataset: 0


In [50]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_4comp.isnull().sum(),
    'Tipo_de_Dato': df_4comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                        Nulos    Tipo_de_Dato
source                      0             str
psi_psa1                    0         float32
psi_psa2                    0         float32
psi_psa3                    0         float32
psi_psa4                    0         float32
psi_tablero                 0         float32
flujo                       0         float32
totalizador                 0         float32
TIME                        0  datetime64[us]
r_psa1                      0            Int8
r_psa2                      0            Int8
r_psa3                      0            Int8
r_psa4                      0            Int8
r_gen1                      0            Int8
r_gen2                      0            Int8
r_bar                       0            Int8
r_sec1                      0            Int8
r_sec2                      0            Int8
r_com1                      0            Int8
r_com2                      0            Int8
r_com3                      0     

In [51]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df_3comp.isnull().sum(),
    'Tipo_de_Dato': df_3comp.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                        Nulos    Tipo_de_Dato
source                      0             str
psi_psa1                    0         float32
psi_psa2                    0         float32
psi_psa3                    0         float32
psi_psa4                    0         float32
psi_tablero                 0         float32
flujo                       0         float32
totalizador                 0         float32
TIME                        0  datetime64[us]
r_psa1                      0            Int8
r_psa2                      0            Int8
r_psa3                      0            Int8
r_psa4                      0            Int8
r_gen1                      0            Int8
r_gen2                      0            Int8
r_bar                       0            Int8
r_sec1                      0            Int8
r_sec2                      0            Int8
r_com1                      0            Int8
r_com2                      0            Int8
r_com3                      0     

## detalles temperatura y revias mas cosas

In [52]:
import pandas as pd

print("--- 1. INSPECCIÓN DE TEMPERATURA ---")
# Vemos las estadísticas de la temperatura para ver su escala (Fahrenheit vs Celsius)
columnas_temp = ['mb_g1_temperatura_f', 'mb_g2_temperatura_f']
print(df_4comp[columnas_temp].describe())

print("\n--- 2. VERIFICACIÓN DE JAULA FALTANTE (SENSOR 10) ---")
# Buscamos un pontón que no tenga jaula 10 y mostramos todas sus variables
columnas_s10 = ['source', 'TIME', 'ox_s10', 'sp_s10', 'm_s10', 'hb_s10']
jaula_10_fantasma = df_4comp[df_4comp['ox_s10'] == -1]

if len(jaula_10_fantasma) > 0:
    print("✅ Primera fila donde la jaula 10 no existe:")
    print(jaula_10_fantasma[columnas_s10].head(1).to_string(index=False))
else:
    print("No se encontraron jaulas 10 en -1.")

print("\n--- 3. VERIFICACIÓN DE FALLAS EN JAULA 1 (SENSOR 1) ---")
# Buscamos si a alguien le falta la jaula 1
columnas_s1 = ['source', 'TIME', 'ox_s1', 'sp_s1', 'm_s1', 'hb_s1']
jaula_1_fallos = df_4comp[df_4comp['ox_s1'] == -1]

print(f"Total de filas con ox_s1 en -1: {len(jaula_1_fallos)}")

if len(jaula_1_fallos) > 0:
    print("\n⚠️ OJO: Existen filas con -1 en la Jaula 1.")
    print("Como todos los pontones tienen Jaula 1, esto NO es una jaula faltante,")
    print("sino un sensor que se averió o desconectó temporalmente.")
    print("Top 5 pontones con sensor 1 malo:")
    print(jaula_1_fallos['source'].value_counts().head(5).to_string())
    
    print("\nEjemplo de esta falla en el sensor 1:")
    print(jaula_1_fallos[columnas_s1].head(3).to_string(index=False))

--- 1. INSPECCIÓN DE TEMPERATURA ---
       mb_g1_temperatura_f  mb_g2_temperatura_f
count         5.549700e+06         5.549700e+06
mean          1.542347e+02         1.762258e+03
std           2.230782e+03         5.118094e+03
min          -3.276800e+04        -3.276800e+04
25%           0.000000e+00         0.000000e+00
50%           0.000000e+00         0.000000e+00
75%           2.000000e+01         2.000000e+01
max           2.457600e+04         1.706900e+04

--- 2. VERIFICACIÓN DE JAULA FALTANTE (SENSOR 10) ---
✅ Primera fila donde la jaula 10 no existe:
source                    TIME  ox_s10  sp_s10  m_s10  hb_s10
 POX15 2026-03-29 17:30:59.999    -1.0     0.0      0     0.0

--- 3. VERIFICACIÓN DE FALLAS EN JAULA 1 (SENSOR 1) ---
Total de filas con ox_s1 en -1: 7414

⚠️ OJO: Existen filas con -1 en la Jaula 1.
Como todos los pontones tienen Jaula 1, esto NO es una jaula faltante,
sino un sensor que se averió o desconectó temporalmente.
Top 5 pontones con sensor 1 malo:
source


In [53]:
import pandas as pd

print("--- ANÁLISIS DE DURACIÓN DE FALLAS EN JAULA 1 ---")

def analizar_cortes_sensor_ox(df, nombre_sistema, columna='ox_s1'):
    print(f"\n>> Analizando {nombre_sistema} (Sensor: {columna})")
    
    # Asegurar orden cronológico por pontón
    df.sort_values(by=['source', 'TIME'], inplace=True)
    
    # En nuestro dataset actual, las fallas aisladas quedaron marcadas con -1.0
    es_falla = df[columna] == -1.0
    
    if not es_falla.any():
        print("No hay fallas registradas en este sensor.")
        return
        
    # Crear un ID único para cada bloque continuo de fallas
    id_bloques = (es_falla != es_falla.shift()).cumsum()
    
    # Agrupar solo las filas que SON falla y contar su duración (1 fila = 1 minuto)
    duracion_cortes = df[es_falla].groupby(id_bloques).size()
    
    print(f"Total de eventos de desconexión: {len(duracion_cortes)}")
    print(f"Duración PROMEDIO del corte: {duracion_cortes.mean():.1f} minutos")
    print(f"Duración MEDIANA del corte: {duracion_cortes.median():.1f} minutos")
    print(f"Duración MÁXIMA (el peor caso): {duracion_cortes.max()} minutos")
    
    print("\nTop 5 cortes más largos (minutos):")
    print(duracion_cortes.sort_values(ascending=False).head(5).to_string(index=False))

# Ejecutamos el análisis para el sistema problemático
analizar_cortes_sensor_ox(df_4comp, "Sistema de 4 Compresores")

--- ANÁLISIS DE DURACIÓN DE FALLAS EN JAULA 1 ---

>> Analizando Sistema de 4 Compresores (Sensor: ox_s1)
Total de eventos de desconexión: 23
Duración PROMEDIO del corte: 322.3 minutos
Duración MEDIANA del corte: 390.0 minutos
Duración MÁXIMA (el peor caso): 480 minutos

Top 5 cortes más largos (minutos):
ox_s1
480
436
432
416
410


In [54]:
def correccion_maestra(df, nombre_sistema):
    print(f"\n⏳ Procesando {nombre_sistema}...")
    
    # 1. Temperatura (sin cambios, está bien)
    cols_temp = ['mb_g1_temperatura_f', 'mb_g2_temperatura_f']
    for col in cols_temp:
        if col in df.columns:
            df.loc[(df[col] < -100) | (df[col] > 1000), col] = np.nan
            df[col] = df.groupby('source')[col].ffill()
            df[col] = df[col].fillna(0)
    
    # 2. Identificar jaulas INEXISTENTES por pontón (máximo histórico <= 0)
    for i in range(1, 13):
        col_ox = f'ox_s{i}'
        col_hb = f'hb_s{i}'
        col_sp = f'sp_s{i}'
        col_m  = f'm_s{i}'
        if col_ox not in df.columns:
            continue
        
        # Máximo histórico de ox por pontón
        max_ox_por_ponton = df.groupby('source')[col_ox].transform('max')
        
        # Caso A: Jaula INEXISTENTE → todo a -1, incluyendo hb
        es_jaula_fantasma = (max_ox_por_ponton <= 0)
        for col in [col_ox, col_hb, col_sp, col_m]:
            if col in df.columns:
                df.loc[es_jaula_fantasma, col] = -1
        
        # Caso B: Sensor fallido temporalmente (ox=-1 pero la jaula SÍ existe)
        # hb debe ser 0 (equipo muerto/desconectado), NO -1
        es_falla_temporal = (~es_jaula_fantasma) & (df[col_ox] == -1)
        for col in [col_sp, col_m]:
            if col in df.columns:
                df.loc[es_falla_temporal, col] = -1  # sp y m también a -1
        if col_hb in df.columns:
            df.loc[es_falla_temporal, col_hb] = 0   # ← ESTE ES EL FIX CLAVE
    
    print("✅ Corrección aplicada con distinción jaula fantasma vs sensor fallido.")
    return df

In [55]:
import pandas as pd

def auditar_heartbeats(df, nombre_sistema):
    # Seleccionamos todas las columnas que empiezan con 'hb_'
    cols_hb = [c for c in df.columns if c.startswith('hb_')]
    
    print(f"\n--- Auditoría de Heartbeats en {nombre_sistema} ---")
    for col in cols_hb:
        valores_unicos = df[col].dropna().unique()
        # Buscamos valores que no sean 0, 1, o nuestro relleno -1
        anomalos = [v for v in valores_unicos if v not in [0.0, 1.0, -1.0]]
        
        if anomalos:
            print(f"⚠️ ¡ALERTA! Columna {col} tiene valores extraños: {anomalos}")

# Corremos la auditoría
auditar_heartbeats(df_4comp, "Sistema de 4 Compresores")
auditar_heartbeats(df_3comp, "Sistema de 3 Compresores")


--- Auditoría de Heartbeats en Sistema de 4 Compresores ---
⚠️ ¡ALERTA! Columna hb_s9 tiene valores extraños: [np.float32(17307.0), np.float32(17306.0)]
⚠️ ¡ALERTA! Columna hb_s10 tiene valores extraños: [np.float32(2026.0)]
⚠️ ¡ALERTA! Columna hb_s11 tiene valores extraños: [np.float32(5.0), np.float32(4.0), np.float32(17357.0), np.float32(17356.0)]
⚠️ ¡ALERTA! Columna hb_s12 tiene valores extraños: [np.int8(13), np.int8(12)]

--- Auditoría de Heartbeats en Sistema de 3 Compresores ---


In [56]:
def saneamiento_radical_hb(df, nombre_sistema):
    cols_hb = [c for c in df.columns if c.startswith('hb_')]
    for col in cols_hb:
        # Convertir a NaN solo los valores que NO sean 0, 1, o -1
        mascara_anomalo = ~df[col].isin([0.0, 1.0, -1.0])
        df.loc[mascara_anomalo, col] = np.nan
        
        # Los NaN que quedaron (valores raros) → los marcamos como 0 (desconocido = muerto)
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype('Int8')
    return df

In [57]:
def auditoria_final(df, nombre):
    print(f"\n--- Auditoría Final: {nombre} ---")
    
    # 1. Verificar si existen valores distintos a 0 o 1 en hb_
    cols_hb = [c for c in df.columns if c.startswith('hb_')]
    for col in cols_hb:
        invalidos = df[~df[col].isin([0, 1])][col].unique()
        if len(invalidos) > 0:
            print(f"⚠️ ¡ALERTA! Columna {col} tiene valores inválidos: {invalidos}")
            
    # 2. Verificar que el oxígeno no tenga nulos (debería ser 0)
    cols_ox = [f'ox_s{i}' for i in range(1, 13)]
    nulos_ox = df[cols_ox].isnull().sum().sum()
    print(f"✅ Nulos en sensores de oxígeno: {nulos_ox}")
    
    # 3. Verificar que la temperatura no tenga nulos ni valores locos
    temp_nulos = df[['mb_g1_temperatura_f', 'mb_g2_temperatura_f']].isnull().sum().sum()
    print(f"✅ Nulos en temperatura: {temp_nulos}")

auditoria_final(df_4comp, "Sistema 4 Compresores")
auditoria_final(df_3comp, "Sistema 3 Compresores")


--- Auditoría Final: Sistema 4 Compresores ---
⚠️ ¡ALERTA! Columna hb_gen1 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_gen2 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb_sec1 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_sec2 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb_com1 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_com2 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb_com3 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_com4 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb_psa1 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_psa2 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb_psa3 tiene valores inválidos: [-1.]
⚠️ ¡ALERTA! Columna hb_psa4 tiene valores inválidos: <IntegerArray>
[-1]
Length: 1, dtype: Int8
⚠️ ¡ALERTA! Columna hb

In [58]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# Colores para la terminal
# ─────────────────────────────────────────────
OK   = "\033[92m✅\033[0m"
ERR  = "\033[91m❌\033[0m"
HEAD = "\033[1;94m"
END  = "\033[0m"

resultados = []

def check(nombre, condicion, detalle=""):
    estado = OK if condicion else ERR
    print(f"  {estado} {nombre}")
    if not condicion and detalle:
        print(f"      → {detalle}")
    resultados.append((nombre, condicion))

# ══════════════════════════════════════════════════════════════
# FUNCIONES A TESTEAR (versiones corregidas)
# ══════════════════════════════════════════════════════════════

def correccion_maestra(df, nombre_sistema):
    cols_temp = ['mb_g1_temperatura_f', 'mb_g2_temperatura_f']
    for col in cols_temp:
        if col in df.columns:
            df.loc[(df[col] < -100) | (df[col] > 1000), col] = np.nan
            df[col] = df.groupby('source')[col].ffill()
            df[col] = df[col].fillna(0)

    for i in range(1, 13):
        col_ox = f'ox_s{i}'
        col_hb = f'hb_s{i}'
        col_sp = f'sp_s{i}'
        col_m  = f'm_s{i}'
        if col_ox not in df.columns:
            continue

        max_ox_por_ponton = df.groupby('source')[col_ox].transform('max')
        es_jaula_fantasma = (max_ox_por_ponton <= 0)

        for col in [col_ox, col_hb, col_sp, col_m]:
            if col in df.columns:
                df.loc[es_jaula_fantasma, col] = -1

        es_falla_temporal = (~es_jaula_fantasma) & (df[col_ox] == -1)
        for col in [col_sp, col_m]:
            if col in df.columns:
                df.loc[es_falla_temporal, col] = -1
        if col_hb in df.columns:
            df.loc[es_falla_temporal, col_hb] = 0

    return df

def saneamiento_radical_hb(df, nombre_sistema):
    cols_hb = [c for c in df.columns if c.startswith('hb_')]
    for col in cols_hb:
        mascara_anomalo = ~df[col].isin([0.0, 1.0, -1.0])
        df.loc[mascara_anomalo, col] = np.nan
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype('Int8')
    return df


# ══════════════════════════════════════════════════════════════
# TEST 1 — Jaula INEXISTENTE (sensor 10 en planta de 8 jaulas)
# ══════════════════════════════════════════════════════════════
print(f"\n{HEAD}TEST 1 — Jaula inexistente (sensor 10, planta de 8 jaulas){END}")
print("  Regla: si el sensor nunca tuvo valor real → ox=-1, hb=-1, sp=-1, m=-1")

df1 = pd.DataFrame({
    'source': ['POX_A'] * 5,
    'TIME':   pd.date_range('2024-01-01', periods=5, freq='min'),
    'ox_s10': [-1.0] * 5,   # Nunca tuvo valor válido
    'hb_s10': [0.0, 1.0, 0.0, 1.0, 0.0],
    'sp_s10': [90.0] * 5,
    'm_s10':  [1.0] * 5,
    'mb_g1_temperatura_f': [72.0] * 5,
    'mb_g2_temperatura_f': [74.0] * 5,
})

df1 = correccion_maestra(df1.copy(), "test1")
check("ox_s10 == -1 (jaula inexistente)",     (df1['ox_s10'] == -1).all())
check("hb_s10 == -1 (no existe, no late)",    (df1['hb_s10'] == -1).all(),
      f"Valores: {df1['hb_s10'].unique()}")
check("sp_s10 == -1",                         (df1['sp_s10'] == -1).all())
check("m_s10  == -1",                         (df1['m_s10']  == -1).all())


# ══════════════════════════════════════════════════════════════
# TEST 2 — Sensor que FALLA en una jaula que SÍ existe
# ══════════════════════════════════════════════════════════════
print(f"\n{HEAD}TEST 2 — Sensor fallido (s1 en planta con jaula 1 real){END}")
print("  Regla: ox=-1 por falla temporal → hb=0 (muerto, no -1), sp=-1, m=-1")

df2 = pd.DataFrame({
    'source': ['POX_B'] * 6,
    'TIME':   pd.date_range('2024-01-01', periods=6, freq='min'),
    'ox_s1':  [8.5, 8.3, -1.0, -1.0, 8.7, 8.6],  # Filas 2-3: falla temporal
    'hb_s1':  [1.0, 1.0,  0.0,  0.0, 1.0, 1.0],
    'sp_s1':  [9.0] * 6,
    'm_s1':   [1.0] * 6,
    'mb_g1_temperatura_f': [70.0] * 6,
    'mb_g2_temperatura_f': [71.0] * 6,
})

df2 = correccion_maestra(df2.copy(), "test2")

# Filas normales (ox > 0)
filas_normales = df2[df2['ox_s1'] > 0]
# Filas con falla
filas_falla    = df2[df2['ox_s1'] == -1]

check("ox_s1 normal no cambia",           (filas_normales['ox_s1'] > 0).all())
check("hb_s1 en falla == 0 (no -1)",      (filas_falla['hb_s1'] == 0).all(),
      f"Valores: {filas_falla['hb_s1'].unique()}")
check("sp_s1 en falla == -1",             (filas_falla['sp_s1'] == -1).all())
check("m_s1  en falla == -1",             (filas_falla['m_s1']  == -1).all())
check("hb_s1 normal sigue siendo 0 o 1",  filas_normales['hb_s1'].isin([0, 1]).all())


# ══════════════════════════════════════════════════════════════
# TEST 3 — Saneamiento de heartbeats con valores anómalos
# ══════════════════════════════════════════════════════════════
print(f"\n{HEAD}TEST 3 — Saneamiento de HB: valores raros (17307, 2026, etc){END}")
print("  Regla: solo quedan 0, 1, -1. Los valores raros → 0 (muerto)")

df3 = pd.DataFrame({
    'source':  ['POX_C'] * 5,
    'TIME':    pd.date_range('2024-01-01', periods=5, freq='min'),
    'hb_com1': [1.0, 0.0, 17307.0, 2026.0, 1.0],  # Valores raros en medio
    'hb_s1':   [1.0, -1.0, 1.0, 0.0, 1.0],         # El -1 es legítimo (jaula fantasma)
})

df3 = saneamiento_radical_hb(df3.copy(), "test3")

check("hb_com1 solo tiene 0 y 1",          df3['hb_com1'].isin([0, 1]).all(),
      f"Valores: {df3['hb_com1'].unique()}")
check("hb_s1 conserva el -1 legítimo",     (-1 in df3['hb_s1'].values),
      f"Valores: {df3['hb_s1'].unique()}")
check("hb_s1 no tiene valores raros",      df3['hb_s1'].isin([0, 1, -1]).all())
check("dtype es Int8",                     str(df3['hb_com1'].dtype) == 'Int8')


# ══════════════════════════════════════════════════════════════
# TEST 4 — Temperatura con códigos de error de hardware
# ══════════════════════════════════════════════════════════════
print(f"\n{HEAD}TEST 4 — Temperatura: códigos de error (-32768, 9999){END}")
print("  Regla: valores < -100 o > 1000 son error de hardware → reemplazar con ffill")

df4 = pd.DataFrame({
    'source':               ['POX_D'] * 6,
    'TIME':                 pd.date_range('2024-01-01', periods=6, freq='min'),
    'ox_s1':                [8.0] * 6,
    'mb_g1_temperatura_f':  [68.0, 69.0, -32768.0, 9999.0, 70.0, 71.0],
    'mb_g2_temperatura_f':  [65.0] * 6,
})

df4 = correccion_maestra(df4.copy(), "test4")

check("No quedan errores de hardware (< -100)",  (df4['mb_g1_temperatura_f'] >= -100).all(),
      f"Valores: {df4['mb_g1_temperatura_f'].values}")
check("No quedan errores de hardware (> 1000)",  (df4['mb_g1_temperatura_f'] <= 1000).all())
check("Temperatura propagada por ffill",
      df4['mb_g1_temperatura_f'].iloc[2] == 69.0,
      f"Esperaba 69.0, obtuvo {df4['mb_g1_temperatura_f'].iloc[2]}")


# ══════════════════════════════════════════════════════════════
# TEST 5 — Dos pontones en el mismo df: uno con jaula, uno sin ella
# ══════════════════════════════════════════════════════════════
print(f"\n{HEAD}TEST 5 — Mixto: POX_E (tiene jaula 5), POX_F (no tiene jaula 5){END}")
print("  Regla: POX_F → hb_s5=-1. POX_E con falla → hb_s5=0")

df5 = pd.DataFrame({
    'source': ['POX_E', 'POX_E', 'POX_E', 'POX_F', 'POX_F', 'POX_F'],
    'TIME':   pd.date_range('2024-01-01', periods=6, freq='min'),
    'ox_s5':  [7.0,  -1.0,  7.5,   -1.0,  -1.0,  -1.0],  # POX_F nunca tuvo valor real
    'hb_s5':  [1.0,   0.0,  1.0,    0.0,   0.0,   0.0],
    'sp_s5':  [9.0,   9.0,  9.0,    9.0,   9.0,   9.0],
    'm_s5':   [1.0,   1.0,  1.0,    1.0,   1.0,   1.0],
    'mb_g1_temperatura_f': [70.0] * 6,
    'mb_g2_temperatura_f': [70.0] * 6,
})

df5 = correccion_maestra(df5.copy(), "test5")

pox_e = df5[df5['source'] == 'POX_E']
pox_f = df5[df5['source'] == 'POX_F']
falla_e = pox_e[pox_e['ox_s5'] == -1]

check("POX_F: hb_s5 == -1 (jaula inexistente)",   (pox_f['hb_s5'] == -1).all(),
      f"Valores: {pox_f['hb_s5'].unique()}")
check("POX_E con falla: hb_s5 == 0 (no -1)",      (falla_e['hb_s5'] == 0).all(),
      f"Valores: {falla_e['hb_s5'].unique()}")
check("POX_E normal: hb_s5 == 1",
      (pox_e[pox_e['ox_s5'] > 0]['hb_s5'] == 1).all())
check("POX_F: sp_s5 == -1",                        (pox_f['sp_s5'] == -1).all())
check("POX_E con falla: sp_s5 == -1",              (falla_e['sp_s5'] == -1).all())


# ══════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ══════════════════════════════════════════════════════════════
aprobados = sum(1 for _, ok in resultados if ok)
total     = len(resultados)
fallidos  = [(n, ok) for n, ok in resultados if not ok]

print(f"\n{'═'*55}")
print(f"  RESULTADO: {aprobados}/{total} tests pasaron")
if fallidos:
    print(f"\n  {ERR} Tests fallidos:")
    for nombre, _ in fallidos:
        print(f"    - {nombre}")
else:
    print(f"  {OK} ¡Toda la lógica es correcta!")
print(f"{'═'*55}\n")


TEST 1 — Jaula inexistente (sensor 10, planta de 8 jaulas)
  Regla: si el sensor nunca tuvo valor real → ox=-1, hb=-1, sp=-1, m=-1
  ✅ ox_s10 == -1 (jaula inexistente)
  ✅ hb_s10 == -1 (no existe, no late)
  ✅ sp_s10 == -1
  ✅ m_s10  == -1

TEST 2 — Sensor fallido (s1 en planta con jaula 1 real)
  Regla: ox=-1 por falla temporal → hb=0 (muerto, no -1), sp=-1, m=-1
  ✅ ox_s1 normal no cambia
  ✅ hb_s1 en falla == 0 (no -1)
  ✅ sp_s1 en falla == -1
  ✅ m_s1  en falla == -1
  ✅ hb_s1 normal sigue siendo 0 o 1

TEST 3 — Saneamiento de HB: valores raros (17307, 2026, etc)
  Regla: solo quedan 0, 1, -1. Los valores raros → 0 (muerto)
  ✅ hb_com1 solo tiene 0 y 1
  ✅ hb_s1 conserva el -1 legítimo
  ✅ hb_s1 no tiene valores raros
  ✅ dtype es Int8

TEST 4 — Temperatura: códigos de error (-32768, 9999)
  Regla: valores < -100 o > 1000 son error de hardware → reemplazar con ffill
  ✅ No quedan errores de hardware (< -100)
  ✅ No quedan errores de hardware (> 1000)
  ✅ Temperatura propagada por 